In [27]:
import pandas as pd

files = {
    'results_historical': '../data/processed/results_historical.csv',
    'fixtures_enriched':  '../data/processed/wc_2026_fixtures_enriched.csv',
    'elo_history':        '../data/processed/elo_history.csv',
}
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

for name, path in files.items():
    df = pd.read_csv(path)
    print(f"\n{'='*50}")
    print(f"{name}: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    print(df.head(3))


results_historical: (49433, 10)
Columns: ['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral', 'outcome']
         date home_team away_team  home_score  away_score tournament     city  \
0  1872-11-30  Scotland   England           0           0   Friendly  Glasgow   
1  1873-03-08   England  Scotland           4           2   Friendly   London   
2  1874-03-07  Scotland   England           2           1   Friendly  Glasgow   

    country  neutral   outcome  
0  Scotland    False      draw  
1   England    False  home_win  
2  Scotland    False  home_win  

fixtures_enriched: (104, 26)
Columns: ['stage', 'group', 'team1', 'team2', 'venue', 'city', 'country', 'date', 'kickoff_et', 'is_placeholder_match', 'team1_confederation', 'team1_fifa_rank', 'team1_coach', 'team1_elo_rating', 'team1_elo_rank', 'team1_elo_country_code', 'team1_elo_is_host', 'team2_confederation', 'team2_fifa_rank', 'team2_coach', 'team2_elo_rating', 'team2_elo_ran

In [28]:
results  = pd.read_csv('../data/processed/results_historical.csv')
fixtures = pd.read_csv('../data/processed/wc_2026_fixtures_enriched.csv')
elo      = pd.read_csv('../data/processed/elo_history.csv')

In [29]:
# Verify team name consistency — DE uses Türkiye / USA / Czechia as canonical
results_teams = set(results['home_team'].unique()) | set(results['away_team'].unique())

# Check only known (non-placeholder) WC 2026 teams
known_fixtures = fixtures[fixtures['is_placeholder_match'] == False]
fixture_teams = set(known_fixtures['team1']) | set(known_fixtures['team2'])

missing = fixture_teams - results_teams
print(f"WC 2026 teams not found in historical results: {len(missing)}")
if missing:
    print(sorted(missing))
else:
    print("All WC 2026 teams found — name convention consistent.")

WC 2026 teams not found in historical results: 0
All WC 2026 teams found — name convention consistent.


In [30]:
# Name standardization already applied by DE (Turkey→Türkiye, United States→USA, Czech Republic→Czechia)
# No mapping needed — both results and fixtures use the same canonical names

# Filter to group stage using is_placeholder_match flag from enriched file
group_stage = fixtures[fixtures['is_placeholder_match'] == False].copy()
print(f"Group stage matches: {len(group_stage)}")
print(group_stage[['team1', 'team2', 'date', 'team1_elo_rating', 'team2_elo_rating']].head(5))

Group stage matches: 72
         team1         team2        date  team1_elo_rating  team2_elo_rating
0       Mexico  South Africa  2026-06-11            1860.0            1524.0
1  South Korea       Czechia  2026-06-11            1752.0            1726.0
2  South Korea        Mexico  2026-06-18            1752.0            1860.0
3      Czechia  South Africa  2026-06-18            1726.0            1524.0
4      Czechia        Mexico  2026-06-24            1726.0            1860.0


In [31]:
# Check tournament types and counts
print(results['tournament'].value_counts().head(20))

# Check date range
print(f"\nDate range: {results['date'].min()} → {results['date'].max()}")

# DE already separated future matches (results_future.csv) — no missing scores here
print(f"\nMissing values:\n{results.isnull().sum()}")

tournament
Friendly                                18388
FIFA World Cup qualification             8771
UEFA Euro qualification                  2824
African Cup of Nations qualification     2327
FIFA World Cup                            992
Copa América                              869
African Cup of Nations                    845
AFC Asian Cup qualification               829
UEFA Nations League                       658
CECAFA Cup                                620
CFU Caribbean Cup qualification           606
Merdeka Tournament                        599
British Home Championship                 523
CONCACAF Nations League                   422
AFC Asian Cup                             421
Gold Cup                                  420
Gulf Cup                                  410
Island Games                              394
UEFA Euro                                 388
Asian Games                               368
Name: count, dtype: int64

Date range: 1872-11-30 → 2026-06-18

Missi

In [32]:
# Keep only competitive matches (DS decision — excludes friendlies and regional cups)
competitive = [
    'FIFA World Cup',
    'FIFA World Cup qualification',
    'UEFA Euro',
    'UEFA Euro qualification',
    'Copa América',
    'African Cup of Nations',
    'AFC Asian Cup',
    'Gold Cup',
    'UEFA Nations League',
    'CONCACAF Nations League',
]

df = results[results['tournament'].isin(competitive)].copy()
print(f"Competitive matches: {len(df)}")

# Use outcome column from DE directly — keeps home_win / draw / away_win convention
df['result'] = df['outcome']
print(f"\nOutcome distribution:\n{df['result'].value_counts(normalize=True).round(3)}")

# World Cup matches only — for EDA comparison
wc = df[df['tournament'] == 'FIFA World Cup']
print(f"\nWorld Cup matches only ({len(wc)}):\n{wc['result'].value_counts(normalize=True).round(3)}")

Competitive matches: 16610

Outcome distribution:
result
home_win    0.492
away_win    0.293
draw        0.216
Name: proportion, dtype: float64

World Cup matches only (992):
result
home_win    0.458
away_win    0.317
draw        0.226
Name: proportion, dtype: float64


In [33]:
# Check neutral flag distribution
print("Neutral flag in competitive matches:")
print(df['neutral'].value_counts())

# Outcome distribution on neutral ground only
neutral_df = df[df['neutral'] == True]
print(f"\nOutcome on neutral ground ({len(neutral_df)} matches):")
print(neutral_df['result'].value_counts(normalize=True).round(3))

# Outcome distribution in WC specifically
print(f"\nNeutral flag in WC matches:")
print(wc['neutral'].value_counts())

Neutral flag in competitive matches:
neutral
False    12596
True      4014
Name: count, dtype: int64

Outcome on neutral ground (4014 matches):
result
home_win    0.422
away_win    0.345
draw        0.234
Name: proportion, dtype: float64

Neutral flag in WC matches:
neutral
True     866
False    126
Name: count, dtype: int64


### Feature Engineering ###

In [34]:
import numpy as np

df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

def get_recent_form(df, team, date, n=10):
    """Get win rate and avg goals for a team in last N matches before date."""
    mask = (
        ((df['home_team'] == team) | (df['away_team'] == team)) &
        (df['date'] < date)
    )
    recent = df[mask].tail(n)
    if len(recent) == 0:
        return pd.Series({'win_rate': 0.5, 'avg_goals_scored': 1.5, 'avg_goals_conceded': 1.5})

    wins, goals_for, goals_against = 0, 0, 0
    for _, row in recent.iterrows():
        if row['home_team'] == team:
            goals_for     += row['home_score']
            goals_against += row['away_score']
            if row['result'] == 'home_win': wins += 1
        else:
            goals_for     += row['away_score']
            goals_against += row['home_score']
            if row['result'] == 'away_win': wins += 1

    return pd.Series({
        'win_rate':           wins / len(recent),
        'avg_goals_scored':   goals_for / len(recent),
        'avg_goals_conceded': goals_against / len(recent),
    })

# Test on one match
test_date = pd.Timestamp('2022-11-20')
print(get_recent_form(df, 'France', test_date, n=10))

win_rate              0.5
avg_goals_scored      2.0
avg_goals_conceded    1.0
dtype: float64


In [35]:
# Prepare Elo - take last snapshot per year per country
elo['snapshot_date'] = pd.to_datetime(elo['snapshot_date'])
elo_latest = elo.sort_values('snapshot_date').groupby(['country', 'year']).last().reset_index()

def get_elo(team, date, elo_df):
    """Get most recent Elo rating for a team before given date"""
    year = date.year
    mask = (elo_df['country'] == team) & (elo_df['year'] < year)
    recent = elo_df[mask]
    if len(recent) == 0:
        return 1500  # default rating
    return recent.sort_values('year').iloc[-1]['rating']

# Test
print(get_elo('France', pd.Timestamp('2022-11-20'), elo_latest))
print(get_elo('Brazil', pd.Timestamp('2022-11-20'), elo_latest))
print(get_elo('Argentina', pd.Timestamp('2022-11-20'), elo_latest))


2114
2149
2101


### Building final dataset ###


In [36]:
from tqdm import tqdm

rows = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    date = row['date']
    home = row['home_team']
    away = row['away_team']

    # Home team form
    home_form = get_recent_form(df, home, date, n=10)
    # Away team form
    away_form = get_recent_form(df, away, date, n=10)

    # Elo ratings
    home_elo = get_elo(home, date, elo_latest)
    away_elo = get_elo(away, date, elo_latest)

    rows.append({
        'date':                    date,
        'home_team':               home,
        'away_team':               away,
        'neutral':                 row['neutral'],
        'tournament':              row['tournament'],
        'home_win_rate':           home_form['win_rate'],
        'home_avg_goals_scored':   home_form['avg_goals_scored'],
        'home_avg_goals_conceded': home_form['avg_goals_conceded'],
        'away_win_rate':           away_form['win_rate'],
        'away_avg_goals_scored':   away_form['avg_goals_scored'],
        'away_avg_goals_conceded': away_form['avg_goals_conceded'],
        'elo_diff':                home_elo - away_elo,
        'home_elo':                home_elo,
        'away_elo':                away_elo,
        'result':                  row['result'],
    })

features_df = pd.DataFrame(rows)
print(f"Dataset shape: {features_df.shape}")
print(features_df.head(3))

100%|██████████| 16610/16610 [03:07<00:00, 88.73it/s]


Dataset shape: (16610, 15)
        date  home_team away_team  neutral    tournament  home_win_rate  \
0 1916-07-02      Chile   Uruguay     True  Copa América            0.5   
1 1916-07-06  Argentina     Chile    False  Copa América            0.5   
2 1916-07-08     Brazil     Chile     True  Copa América            0.5   

   home_avg_goals_scored  home_avg_goals_conceded  away_win_rate  \
0                    1.5                      1.5            0.5   
1                    1.5                      1.5            0.0   
2                    1.5                      1.5            0.0   

   away_avg_goals_scored  away_avg_goals_conceded  elo_diff  home_elo  \
0                    1.5                      1.5      -413      1500   
1                    0.0                      4.0       440      1940   
2                    0.5                      5.0       410      1910   

   away_elo    result  
0      1913  away_win  
1      1500  home_win  
2      1500      draw  


In [37]:
features_df.to_csv('../data/processed/features.csv', index=False)
print(f"Saved: {features_df.shape}")
print(f"\nMissing values:\n{features_df.isnull().sum()}")
print(f"\n{features_df.describe().round(2)}")

Saved: (16610, 15)

Missing values:
date                       0
home_team                  0
away_team                  0
neutral                    0
tournament                 0
home_win_rate              0
home_avg_goals_scored      0
home_avg_goals_conceded    0
away_win_rate              0
away_avg_goals_scored      0
away_avg_goals_conceded    0
elo_diff                   0
home_elo                   0
away_elo                   0
result                     0
dtype: int64

                             date  home_win_rate  home_avg_goals_scored  \
count                       16610       16610.00               16610.00   
mean   2002-01-16 15:19:18.675496           0.39                   1.43   
min           1916-07-02 00:00:00           0.00                   0.00   
25%           1992-10-14 00:00:00           0.20                   0.90   
50%           2005-11-12 00:00:00           0.40                   1.40   
75%           2017-10-10 00:00:00           0.57                 